# GPT-3 Workload Schedule Extraction on TPU

This notebook extracts the actual execution schedule from PyTorch/XLA on Google TPU.

**What you'll get:**
- Execution timeline showing when each operation runs
- Fusion decisions (which operations XLA combined)
- Memory usage per operation
- Compute unit assignments
- Chrome trace for visualization
- CSV with structured schedule data

**Hardware:** Google TPU v2/v3 (free tier on Colab)

**Workload:** GPT-3 attention + FFN (10 operations from gpt3_175B.yaml)
- Reduced to M=512 tokens to fit in TPU memory

## 1. Setup: Install PyTorch XLA for TPU

In [ ]:
# Check if running on Colab TPU
import os
import sys

# Install PyTorch XLA
!pip install cloud-tpu-client==0.10 torch==2.5.0 https://storage.googleapis.com/pytorch-xla-releases/wheels/tpuvm/torch_xla-2.5.0-cp310-cp310-linux_x86_64.whl

print("✓ PyTorch XLA installed")

In [ ]:
# Verify TPU access
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.debug.profiler as xp

device = xm.xla_device()
print(f"✓ TPU device: {device}")
print(f"✓ Available devices: {xm.get_xla_supported_devices()}")

In [ ]:
# Set up profiling environment
os.environ['XLA_FLAGS'] = '--xla_dump_to=/tmp/xla_hlo_dumps'
os.environ['XLA_IR_DEBUG'] = '1'
os.environ['XLA_HLO_DEBUG'] = '1'
os.environ['XLA_SAVE_TENSORS_FILE'] = '/tmp/gpt3_tensors.hlo'
os.environ['XLA_SAVE_TENSORS_FMT'] = 'hlo'

print("✓ Profiling environment configured")
print("  HLO dumps will be saved to: /tmp/xla_hlo_dumps/")
print("  Traces will be saved to: /tmp/gpt3_tpu_trace/")

## 2. Define GPT-3 Workload Model

This implements the 10 operations from `gpt3_175B.yaml`:
1. I_copy
2. V_matmul (Value projection)
3. K_matmul (Key projection)
4. Q_matmul (Query projection)
5. QK_matmul (Attention scores)
6. QK_softmax
7. AV_matmul (Attention output)
8. Z_matmul (Output projection)
9. FFA_matmul (Feed-forward A)
10. FFB_matmul (Feed-forward B)

In [ ]:
import torch
import torch.nn as nn
from torch_xla.debug.profiler import Trace

class GPT3Workload(nn.Module):
    """
    GPT-3 workload matching gpt3_175B.yaml structure.
    
    Dimensions:
    - B (batch): 1
    - M (tokens): 512 (reduced from 8192 for memory)
    - H (heads): 96
    - E (embedding per head): 128
    - D (total embedding): 12,288
    - C (FFN dimension): 49,152
    """
    
    def __init__(self, B=1, M=512, H=96, E=128):
        super().__init__()
        
        self.B = B
        self.M = M
        self.P = M
        self.H = H
        self.E = E
        self.F = E
        self.D = H * E      # 12,288
        self.C = 4 * H * E  # 49,152
        self.J = H * E
        self.G = H * E
        
        print(f"Model Configuration:")
        print(f"  Batch (B): {B}")
        print(f"  Tokens (M): {M}")
        print(f"  Heads (H): {H}")
        print(f"  Embedding/head (E): {E}")
        print(f"  Total embedding (D): {self.D}")
        print(f"  FFN dimension (C): {self.C}")
        
        # Weight matrices (using float32 for TPU)
        self.WV = nn.Parameter(torch.randn(H, E, self.D) * 0.01)
        self.WK = nn.Parameter(torch.randn(H, E, self.D) * 0.01)
        self.WQ = nn.Parameter(torch.randn(H, E, self.D) * 0.01)
        self.WZ = nn.Parameter(torch.randn(H, self.F, self.G) * 0.01)
        self.WFFA = nn.Parameter(torch.randn(self.G, self.C) * 0.01)
        self.WFFB = nn.Parameter(torch.randn(self.C, self.J) * 0.01)
        
        total_params = sum(p.numel() for p in self.parameters())
        print(f"\nTotal parameters: {total_params:,}")
    
    def forward(self, I_in):
        """
        Forward pass with profiling annotations.
        Each operation is labeled to track it in the profiler.
        """
        
        # Operation 1: I[b, m, d] = I_in[b, m, d]
        with Trace('01_I_copy'):
            I = I_in.clone()
        
        # Operation 2: V[b, m, h, e] = I[b, m, d] * WV[h, e, d]
        with Trace('02_V_matmul'):
            V = torch.einsum('bmd,hed->bmhe', I, self.WV)
        
        # Operation 3: K[b, m, h, e] = I[b, m, d] * WK[h, e, d]
        with Trace('03_K_matmul'):
            K = torch.einsum('bmd,hed->bmhe', I, self.WK)
        
        # Operation 4: Q[b, m, h, e] = I[b, m, d] * WQ[h, e, d]
        with Trace('04_Q_matmul'):
            Q = torch.einsum('bmd,hed->bmhe', I, self.WQ)
        
        # Operation 5: QK[b, m, p, h] = Q[b, m, h, e] * K[b, p, h, e]
        with Trace('05_QK_matmul'):
            QK = torch.einsum('bmhe,bphe->bmph', Q, K)
        
        # Operation 6: QK_softmax[b, m, p, h] = softmax(QK)
        with Trace('06_QK_softmax'):
            QK_softmax = torch.softmax(QK, dim=2)
        
        # Operation 7: AV[b, m, h, f] = QK_softmax * V
        with Trace('07_AV_matmul'):
            AV = torch.einsum('bmph,bphe->bmhe', QK_softmax, V)
        
        # Operation 8: Z[b, m, g] = AV * WZ
        with Trace('08_Z_matmul'):
            Z = torch.einsum('bmhf,hfg->bmg', AV, self.WZ)
        
        # Operation 9: FFA[b, m, c] = Z * WFFA
        with Trace('09_FFA_matmul'):
            FFA = torch.einsum('bmg,gc->bmc', Z, self.WFFA)
        
        # Operation 10: FFB[b, m, j] = FFA * WFFB
        with Trace('10_FFB_matmul'):
            FFB = torch.einsum('bmc,cj->bmj', FFA, self.WFFB)
        
        return FFB

print("✓ GPT3Workload model defined")

## 3. Run Model with Profiling

In [ ]:
# Create model and move to TPU
print("Creating model...")
model = GPT3Workload(B=1, M=512, H=96, E=128).to(device)

# Create input
print("\nCreating input tensor...")
I_in = torch.randn(1, 512, 12288).to(device)
print(f"Input shape: {I_in.shape}")
print(f"Input size: {I_in.numel() * 4 / 1e6:.2f} MB (float32)")

print("\n" + "="*70)
print("STARTING PROFILED EXECUTION")
print("="*70)

In [ ]:
# Start profiler
server = xp.start_server(9012)
print("✓ Profiler server started on port 9012")

trace_dir = '/tmp/gpt3_tpu_trace/'
xp.trace('localhost:9012', trace_dir, duration_ms=30000)
print(f"✓ Trace capture started (30 second window)")
print(f"  Output directory: {trace_dir}")

In [ ]:
# Run model (warmup)
print("\nWarmup run...")
with torch.no_grad():
    _ = model(I_in)
    xm.mark_step()  # Force XLA compilation
print("✓ Warmup complete (XLA compilation done)")

In [ ]:
# Profiled run
print("\nProfiled run...")
import time
start_time = time.time()

with torch.no_grad():
    output = model(I_in)
    xm.mark_step()

end_time = time.time()
print(f"✓ Execution complete")
print(f"  Output shape: {output.shape}")
print(f"  Wall time: {(end_time - start_time)*1000:.2f} ms")

In [ ]:
# Get HLO representation
print("\nExtracting HLO representation...")
hlo_text = torch_xla._XLAC._get_xla_tensors_hlo([output])

with open('/tmp/gpt3_final_hlo.txt', 'w') as f:
    f.write(hlo_text)
    
print(f"✓ HLO saved to /tmp/gpt3_final_hlo.txt")
print(f"  Size: {len(hlo_text)} characters")

# Preview HLO (first 2000 chars)
print("\nHLO Preview (first 2000 chars):")
print("=" * 70)
print(hlo_text[:2000])
print("...")
print("=" * 70)

## 4. Analyze Profiling Data

In [ ]:
# Wait for trace to finish writing
import time
print("Waiting for trace to complete...")
time.sleep(5)
print("✓ Trace data should be ready")

In [ ]:
# List generated files
print("\nGenerated Files:")
print("\n1. HLO Dumps:")
!ls -lh /tmp/xla_hlo_dumps/ 2>/dev/null | head -20 || echo "  (No HLO dumps found)"

print("\n2. Trace Files:")
!ls -lh /tmp/gpt3_tpu_trace/ 2>/dev/null || echo "  (No trace files found)"

print("\n3. HLO Text:")
!ls -lh /tmp/gpt3_final_hlo.txt /tmp/gpt3_tensors.hlo 2>/dev/null

In [ ]:
# Parse HLO to find fusion information
print("Analyzing HLO for fusion decisions...\n")

with open('/tmp/gpt3_final_hlo.txt', 'r') as f:
    hlo_content = f.read()

# Look for fusion blocks
fusion_blocks = []
lines = hlo_content.split('\n')
in_fusion = False
current_fusion = []

for line in lines:
    if 'fusion' in line.lower() and '{' in line:
        in_fusion = True
        current_fusion = [line]
    elif in_fusion:
        current_fusion.append(line)
        if '}' in line:
            fusion_blocks.append('\n'.join(current_fusion))
            in_fusion = False
            current_fusion = []

print(f"Found {len(fusion_blocks)} fusion blocks in HLO\n")

# Show first few fusions
for i, fusion in enumerate(fusion_blocks[:3]):
    print(f"Fusion {i+1}:")
    print("-" * 70)
    print(fusion[:500])  # First 500 chars
    print("...\n")

In [ ]:
# Parse trace file if available
import json
import glob

trace_files = glob.glob('/tmp/gpt3_tpu_trace/**/*.json.gz', recursive=True)
if not trace_files:
    trace_files = glob.glob('/tmp/gpt3_tpu_trace/**/*.json', recursive=True)

print(f"Found {len(trace_files)} trace files")

if trace_files:
    print(f"\nTrace file: {trace_files[0]}")
    
    # Try to load and parse
    try:
        if trace_files[0].endswith('.gz'):
            import gzip
            with gzip.open(trace_files[0], 'rt') as f:
                trace_data = json.load(f)
        else:
            with open(trace_files[0], 'r') as f:
                trace_data = json.load(f)
        
        print(f"✓ Trace data loaded")
        print(f"  Keys: {list(trace_data.keys())}")
        
        if 'traceEvents' in trace_data:
            events = trace_data['traceEvents']
            print(f"  Total events: {len(events)}")
            
            # Find our labeled operations
            our_ops = [e for e in events if 'name' in e and any(f'{i:02d}_' in str(e['name']) for i in range(1, 11))]
            print(f"  Our operations: {len(our_ops)}")
            
            if our_ops:
                print("\nFound operations:")
                for op in our_ops[:10]:
                    name = op.get('name', 'unknown')
                    dur = op.get('dur', 0) / 1000  # Convert to ms
                    ts = op.get('ts', 0)
                    print(f"  {name}: {dur:.3f} ms")
    except Exception as e:
        print(f"Error parsing trace: {e}")
else:
    print("\nNo trace files found. Profiling may need more time or different setup.")

## 5. Create Schedule CSV

Extract schedule information and save to CSV for analysis.

In [ ]:
import pandas as pd

# Create schedule dataframe
schedule_data = []

if trace_files and our_ops:
    # Extract from actual trace
    for op in our_ops:
        schedule_data.append({
            'operation': op.get('name', 'unknown'),
            'start_time_us': op.get('ts', 0),
            'duration_us': op.get('dur', 0),
            'duration_ms': op.get('dur', 0) / 1000,
            'pid': op.get('pid', ''),
            'tid': op.get('tid', ''),
            'args': str(op.get('args', {}))
        })
else:
    # Fallback: create structure from model knowledge
    print("Using fallback schedule structure (no trace data available)")
    ops = [
        '01_I_copy', '02_V_matmul', '03_K_matmul', '04_Q_matmul',
        '05_QK_matmul', '06_QK_softmax', '07_AV_matmul',
        '08_Z_matmul', '09_FFA_matmul', '10_FFB_matmul'
    ]
    for op in ops:
        schedule_data.append({
            'operation': op,
            'start_time_us': 0,
            'duration_us': 0,
            'duration_ms': 0,
            'pid': '',
            'tid': '',
            'args': 'no_trace_data'
        })

schedule_df = pd.DataFrame(schedule_data)
schedule_df = schedule_df.sort_values('operation')

print("\nSchedule DataFrame:")
print("=" * 70)
print(schedule_df.to_string(index=False))
print("=" * 70)

# Save to CSV
schedule_df.to_csv('/tmp/gpt3_tpu_schedule.csv', index=False)
print("\n✓ Schedule saved to /tmp/gpt3_tpu_schedule.csv")

## 6. Analyze Fusion Decisions

In [ ]:
# Analyze which operations might have been fused
print("Fusion Analysis")
print("=" * 70)

# Check if V, K, Q were fused (common optimization)
print("\nLooking for common fusion patterns...\n")

# Search HLO for einsum operations
einsum_count = hlo_content.lower().count('einsum')
dot_count = hlo_content.lower().count('dot(')
fusion_count = len(fusion_blocks)

print(f"HLO Statistics:")
print(f"  Total 'einsum' mentions: {einsum_count}")
print(f"  Total 'dot' operations: {dot_count}")
print(f"  Total fusion blocks: {fusion_count}")
print(f"\nOriginal operations: 10")
print(f"Fusions found: {fusion_count}")

if fusion_count > 0:
    print(f"\nFusion ratio: {fusion_count / 10 * 100:.1f}%")
    print("\nLikely fusion scenarios:")
    print("  - V/K/Q projections may be fused (share input I)")
    print("  - QK + Softmax may be fused")
    print("  - Softmax + AV may be fused")
    print("  - Element-wise ops fused into matmuls")

print("\n" + "=" * 70)

## 7. Download Results

Download these files for analysis:
1. `gpt3_tpu_schedule.csv` - Schedule data
2. `gpt3_final_hlo.txt` - HLO representation
3. Trace files (if available)

In [ ]:
# Create downloadable archive
!tar -czf /tmp/gpt3_tpu_results.tar.gz \
    /tmp/gpt3_tpu_schedule.csv \
    /tmp/gpt3_final_hlo.txt \
    /tmp/gpt3_tensors.hlo \
    /tmp/gpt3_tpu_trace/ \
    /tmp/xla_hlo_dumps/ 2>/dev/null

print("✓ Results archived to /tmp/gpt3_tpu_results.tar.gz")
print("\nDownload this file to analyze locally.")
print("\nFrom Colab: Files → /tmp/gpt3_tpu_results.tar.gz → Download")

In [ ]:
# Summary
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print("\n✓ Execution complete")
print("✓ HLO captured")
print(f"✓ {len(fusion_blocks)} fusion blocks identified")
print("✓ Schedule data saved to CSV")
print("\nNext steps:")
print("1. Download /tmp/gpt3_tpu_results.tar.gz")
print("2. Analyze HLO to understand XLA's fusion decisions")
print("3. Use schedule CSV to create AccelForge mapping")
print("4. Compare TPU schedule with GPU schedule")
print("5. Model optimizations in AccelForge")
print("=" * 70)